# Axon SFT — Host RAM Leak Diagnostic

**Goal:** find where host RAM grows during SFT training on Colab. The GPU-side fixes hold (GPU RAM stays flat at ~24 GB) but the kernel still gets OOM-killed by Linux after ~10 minutes because system RAM climbs past the host limit. This notebook runs SFT long enough to reproduce the climb with **live host-RAM probes** on every section of every training step, plus explicit probes around each eval cycle.

What this notebook does:

1. Pulls latest code from GitHub
2. Mounts Drive and links datasets
3. Builds an SFT config with `batch_size=16`, `num_epochs=10`, `eval_every_n_epochs=5`
4. Uses the full archcad400k dataset (not a tiny slice)
5. Runs training until either (a) it completes 10 epochs cleanly or (b) host RAM OOMs

Each training step prints one `host_rss=X.XX GB` value per section. Each epoch boundary prints the same. Eval start and end print before+after so you can see the jump. Watch the `host_rss` column and tell me which section / phase it climbs at.

**Do not edit anything.** Just run top-to-bottom on a GPU runtime. If it crashes, paste everything from the first `step 0 start:` line down through the end of the output.

## 1. Pull latest code + install deps

In [ ]:
# Pull latest from GitHub
import os

if os.path.exists("Axon"):
    %cd Axon
    !git pull origin main
else:
    !git clone https://github.com/tylermark/Axon.git
    %cd Axon

!pip install -q pydantic pymupdf scipy networkx wandb timm 2>/dev/null

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_mem_gb = (getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU:     {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB)")

## 2. Mount Drive + link datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/axon"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR = f"{DRIVE_ROOT}/datasets"

import os
for d in [CKPT_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

# Link archcad400k into the working directory
from pathlib import Path
local_data = Path("datasets")
local_data.mkdir(exist_ok=True)

src = Path(DATA_DIR) / "archcad400k"
target = local_data / "archcad400k"
if src.exists() and not target.exists():
    os.symlink(str(src), str(target))

assert src.exists(), f"archcad400k not found at {src} — upload it first"
print(f"Dataset linked: {target} -> {src}")

## 3. Build SFT config matching the main notebook (batch=16, eval=5)

In [ ]:
import torch
from src.training.sft import SFTTrainer, SFTConfig

# Match the main notebook's shape so the leak reproduces, but short
# enough to crash within a few minutes if it's going to crash.
sft_config = SFTConfig(
    num_epochs=10,
    batch_size=16,
    learning_rate=5e-5,
    checkpoint_dir="/tmp/sft_diag",  # throwaway — do not pollute Drive
    device="cuda" if torch.cuda.is_available() else "cpu",
    wandb_enabled=False,           # no W&B login prompt
    eval_every_n_epochs=5,         # match real notebook
    save_every_n_epochs=999,       # skip checkpoint saves
    num_workers=0,
    profile_first_n_steps=999_999, # profile EVERY step so we see growth
)

print(f"Batch size:   {sft_config.batch_size}")
print(f"Epochs:       {sft_config.num_epochs}")
print(f"Eval every:   {sft_config.eval_every_n_epochs} epochs")
print(f"Device:       {sft_config.device}")
print(f"Profile:      every step")

## 4. Build models + tiny dataset

In [ ]:
import torch
from src.pipeline.config import AxonConfig
from src.tokenizer.cross_attention import Tokenizer
from src.diffusion.reverse import GraphDiffusionModel
from src.constraints.sat_solver import ConstraintSolver
from src.training.sft_data import SFTDatasetAdapter
from src.training.data_engine import ArchCAD400KDataset

# ── Models ──
config = AxonConfig()
tokenizer_model = Tokenizer(config=config.tokenizer)
diffusion_model = GraphDiffusionModel(config=config.diffusion)
constraint_module = ConstraintSolver(config=config.constraints)

param_count = sum(
    sum(p.numel() for p in m.parameters())
    for m in [tokenizer_model, diffusion_model, constraint_module]
)
print(f"Total params: {param_count:,}")

# ── Dataset (full, with 90/10 train/eval split matching the main notebook) ──
base_dataset = ArchCAD400KDataset(data_root="datasets/")
full_dataset = SFTDatasetAdapter(base_dataset, max_nodes=512)
print(f"Full dataset: {len(full_dataset)} samples")

n_total = len(full_dataset)
n_eval = max(1, n_total // 10)
n_train = n_total - n_eval
train_dataset, eval_dataset = torch.utils.data.random_split(
    full_dataset, [n_train, n_eval],
    generator=torch.Generator().manual_seed(42),
)
print(f"Split: {n_train} train, {n_eval} eval")

## 5. Run SFT with live timing + host RAM probes

Each training step will print one line per section like:

```
step 0   tokenizer_fwd           42 ms   gpu_alloc=3.10GB   gpu_peak=3.42GB   host_rss=2.15GB
step 0   diffusion_fwd          201 ms   gpu_alloc=14.5GB   gpu_peak=18.2GB   host_rss=2.16GB
```

At epoch boundaries you'll see:

```
Epoch 1/10  loss=...  (...s)  host_rss=2.18GB
```

At eval time:

```
Eval start (epoch 5)  host_rss=2.45GB
Eval done  (180.3s)  host_rss=65.00GB   <-- this jump would be the leak
```

When it crashes OR completes, find the line where `host_rss` jumps the most. That's the leak.

**Use eval_dataset** so eval actually runs at epoch 5.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

sft_trainer = SFTTrainer(
    tokenizer_model=tokenizer_model,
    diffusion_model=diffusion_model,
    constraint_module=constraint_module,
    dataset=train_dataset,
    eval_dataset=eval_dataset,  # ← so eval runs at epoch 5
    config=sft_config,
)

sft_trainer.train()
print("\nDiagnostic run complete.")